# Session 4 — Your turn: Three transformer layouts

The three layouts differ in who may read whom, and in what the model is trained to recover. Write both as small functions.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np
from aij_check import check, close, summary

## Task 1: Three masks

Write `mask_for(layout, n_source, n_target=0)` returning a boolean matrix where True means "may read": for `"encoder"` an `n_source x n_source` matrix of all True; for `"decoder"` an `n_source x n_source` lower-triangular matrix;
for `"cross"` an `n_target x n_source` matrix of all True (every target token reads the whole source). Then fill `allowed_pairs` with the number of True entries for an encoder, a decoder and a cross mask over 6 source tokens (and 4 target tokens for cross).

In [ ]:
def mask_for(layout, n_source, n_target=0):
    ...


allowed_pairs = {"encoder": ..., "decoder": ..., "cross": ...}

In [ ]:
check("the encoder lets every token read every token (36 pairs)", lambda: allowed_pairs["encoder"] == 36)
check("the decoder lets each token read itself and earlier ones (21 pairs)", lambda: allowed_pairs["decoder"] == 21)
check("cross-attention is target x source (24 pairs)", lambda: allowed_pairs["cross"] == 24)
check("a decoder mask has nothing above the diagonal", lambda: not np.triu(mask_for("decoder", 5), 1).any())
summary()

## Task 2: BERT's 80 / 10 / 10 corruption

Write `corrupt(ids, rng, vocab_size=1000, mask_id=0)`: choose each position with probability 0.15; of the chosen positions, replace 80% by `mask_id`, 10% by a random id from `1` to `vocab_size - 1`, and leave 10% unchanged. Return `(corrupted, chosen)`, where `chosen` is a boolean array marking the positions the loss is computed on.

In [ ]:
def corrupt(ids, rng, vocab_size=1000, mask_id=0):
    ...


ids = np.arange(1, 20001) % 999 + 1
corrupted, chosen = corrupt(ids, np.random.default_rng(0))

In [ ]:
check("about 15% of positions are chosen", lambda: abs(chosen.mean() - 0.15) < 0.01)
check("about 80% of the chosen positions become the mask", lambda: abs((corrupted[chosen] == 0).mean() - 0.8) < 0.02)
check("positions that were not chosen are untouched", lambda: np.array_equal(corrupted[~chosen], ids[~chosen]))
check("about 10% of the chosen positions are left unchanged or re-drawn to the same id", lambda: 0.08 < (corrupted[chosen] == ids[chosen]).mean() < 0.13)
summary()